In [5]:
import requests
import json

# Define the bounding box for Matagorda Bay area
min_lon, min_lat = -96.2, 28.3
max_lon, max_lat = -95.9, 28.7

# USGS Earth Explorer API endpoint and parameters
# You need an Earth Explorer account and API key to access this data
api_url = "https://m2m.cr.usgs.gov/api/api/json/stable/"
api_key = "YOUR_API_KEY"  # Replace with your USGS Earth Explorer API key

# Define the area of interest and dataset
params = {
    "datasetName": "LIDAR",
    "spatialFilter": {
        "filterType": "mbr",
        "lowerLeft": {"latitude": min_lat, "longitude": min_lon},
        "upperRight": {"latitude": max_lat, "longitude": max_lon}
    },
    "maxResults": 10,
    "startingNumber": 1,
    "sortOrder": "ASC",
    "apiKey": api_key
}

# Function to send a request to the USGS API
def send_request(endpoint, params):
    headers = {"Content-Type": "application/json"}
    response = requests.post(f"{api_url}{endpoint}", headers=headers, data=json.dumps(params))
    return response.json()

# Login to the API (required for download)
login_params = {
    "username": "YOUR_USERNAME",  # Replace with your USGS Earth Explorer username
    "password": "YOUR_PASSWORD"   # Replace with your USGS Earth Explorer password
}
login_response = send_request("login", login_params)
if "error" in login_response:
    print(f"Login failed: {login_response['error']}")
else:
    api_key = login_response["data"]

    # Search for available scenes
    search_response = send_request("scene-search", params)
    if "error" in search_response:
        print(f"Search failed: {search_response['error']}")
    else:
        scenes = search_response["data"]["results"]
        for scene in scenes:
            download_url = scene["downloadUrl"]
            filename = download_url.split('/')[-1]
            print(f'Downloading {filename}...')
            download_response = requests.get(download_url)
            with open(filename, 'wb') as f:
                f.write(download_response.content)
            print(f'{filename} downloaded successfully.')

    # Logout to end the session
    send_request("logout", {"apiKey": api_key})

print('All files downloaded.')


TypeError: 'NoneType' object is not subscriptable